# GAN Video Enhancer — Kaggle GPU worker

Polls the Heroku API, downloads the queued clip, enhances it with pretrained Real-ESRGAN (x4), uploads the result and reports back.

## Session setup (do this once)
1. Notebook settings → **Accelerator = GPU (T4 / P100)**
2. Notebook settings → **Internet = On** (requires phone-verified Kaggle account)
3. Add-ons → Secrets → add `HEROKU_URL` (e.g. `https://gan-video-demo-xxxx.herokuapp.com`) and `WORKER_SECRET` (same value as the Heroku config var)
4. Run all cells, keep the tab open. **Stop the session when done — never leave the GPU idle.**

The worker only ever sees `HEROKU_URL` and `WORKER_SECRET`. B2 keys stay on Heroku; it uses presigned URLs handed out by the API.

In [ ]:
!git clone https://github.com/xinntao/Real-ESRGAN.git
%cd /kaggle/working/Real-ESRGAN
!pip install -q basicsr facexlib gfpgan realesrgan
!pip install -q -r requirements.txt
!python setup.py develop

In [ ]:
import glob, pathlib
patterns = [
    '/usr/local/lib/python*/dist-packages/basicsr/data/degradations.py',
    '/opt/conda/lib/python*/site-packages/basicsr/data/degradations.py',
]
old = 'from torchvision.transforms.functional_tensor import rgb_to_grayscale'
new = 'from torchvision.transforms.functional import rgb_to_grayscale'
for pat in patterns:
    for hit in glob.glob(pat):
        p = pathlib.Path(hit)
        s = p.read_text()
        if old in s:
            p.write_text(s.replace(old, new))
            print('patched', hit)
print('basicsr fix done')

In [ ]:
import os, time, subprocess, requests

try:
    from kaggle_secrets import UserSecretsClient
    HEROKU_URL = UserSecretsClient().get_secret('HEROKU_URL').rstrip('/')
    WORKER_SECRET = UserSecretsClient().get_secret('WORKER_SECRET')
except Exception:
    HEROKU_URL = os.environ.get('HEROKU_URL', '').rstrip('/')
    WORKER_SECRET = os.environ.get('WORKER_SECRET', '')

HEADERS = {'x-worker-secret': WORKER_SECRET}
MAX_DURATION_S = 10.5
POLL_INTERVAL_S = 5

assert HEROKU_URL and WORKER_SECRET, 'Set HEROKU_URL and WORKER_SECRET (Secrets or env)'
print('worker ready, polling', HEROKU_URL)

In [ ]:
def video_duration(path):
    try:
        out = subprocess.run(
            ['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
             '-of', 'default=noprint_wrappers=1:nokey=1', path],
            capture_output=True, text=True, timeout=30, check=True,
        )
        return float(out.stdout.strip())
    except Exception:
        import cv2
        cap = cv2.VideoCapture(path)
        fps = cap.get(cv2.CAP_PROP_FPS) or 30
        frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
        cap.release()
        return frames / fps

def claim_job():
    r = requests.get(f'{HEROKU_URL}/api/worker/next', headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.json().get('job')

def report_done(job_id, seconds):
    requests.post(f'{HEROKU_URL}/api/worker/{job_id}/done', headers=HEADERS,
                  json={'processingSeconds': round(seconds, 1)}, timeout=30)

def report_failed(job_id, message):
    requests.post(f'{HEROKU_URL}/api/worker/{job_id}/failed', headers=HEADERS,
                  json={'message': str(message)[:500]}, timeout=30)

def process(job):
    job_id = job['jobId']
    started = time.time()
    print(f'[{job_id}] claimed job', flush=True)

    resp = requests.get(job['inputUrl'], timeout=120)
    resp.raise_for_status()
    with open('in.mp4', 'wb') as f:
        f.write(resp.content)
    print(f'[{job_id}] downloaded {len(resp.content)} bytes', flush=True)

    duration = video_duration('in.mp4')
    if duration > MAX_DURATION_S:
        raise ValueError(f'clip is {duration:.1f}s, limit is {MAX_DURATION_S}s')
    print(f'[{job_id}] duration {duration:.1f}s ok', flush=True)

    subprocess.run(
        ['python', 'inference_realesrgan_video.py', '-n', 'realesr-general-x4v3',
         '-i', 'in.mp4', '-o', 'out_dir', '-s', '4', '--suffix', 'out'],
        check=True, capture_output=True, text=True,
    )

    candidates = sorted(glob.glob('out_dir/*.mp4'), key=lambda p: -__import__('os').path.getsize(p))
    if not candidates:
        raise RuntimeError('no output file produced')
    out_path = candidates[0]
    print(f'[{job_id}] enhancement finished: {out_path}', flush=True)

    with open(out_path, 'rb') as f:
        put = requests.put(job['outputUrl'], data=f.read(), timeout=300)
    put.raise_for_status()

    elapsed = time.time() - started
    report_done(job_id, elapsed)
    print(f'[{job_id}] DONE in {elapsed:.1f}s', flush=True)

while True:
    try:
        job = claim_job()
        if not job:
            print('waiting for jobs...', flush=True)
            time.sleep(POLL_INTERVAL_S)
            continue
        try:
            process(job)
        except Exception as e:
            print('job failed:', e, flush=True)
            report_failed(job['jobId'], e)
    except requests.RequestException as e:
        print('api error, retrying:', e, flush=True)
        time.sleep(POLL_INTERVAL_S)
    except KeyboardInterrupt:
        break

## Notes
- Feed the demo clips at 360p–480p: a 10 s, 30 fps clip is ~300 frames and usually takes a few minutes on a free T4/P100.
- Every job logs its processing time — record it for the Test slide together with input/output resolution and GPU model.
- If `basicsr` still complains after the patch cell, restart the kernel and re-run the cells in order.
- Optional stretch: run RealBasicVSR as a second model on the same loop if the first runs go smoothly.
- Stop the session when finished: **never leave the GPU on idle**.